# Remember · Kaggle T4×2

Internet와 GPU T4×2를 활성화하고 수정본을 `/kaggle/working/GRT`에 준비합니다.
RMT → GRT 순서로 두 GPU를 함께 사용합니다. 원본 global batch512, FP32,
train100만 / validation1000 / test10000, 1→2→3→5→10→20→40쌍 중 시간 내 완료 가능한 공통 범위를 선택합니다.
모델별 학습6.5시간 / 사전 측정·평가 포함7시간, 두 모델 전체15시간 상한입니다.
원본 단계별 업데이트 예산을 유지합니다. 기본 예상은20쌍까지이며 실측에 따라 최대40쌍입니다.
고정 길이 EM99%는 기능 성공 지표로 기록하고 조기 전환에는 사용하지 않습니다.
출력은 `/kaggle/working`에 남습니다. Run All 후 Save Version의 output으로 보존하세요.

In [ ]:
from pathlib import Path
import os, sys, subprocess, signal, time, json

REPO_DIR = Path('/kaggle/working/GRT')
RUN_DIR = Path('/kaggle/working/remember_t4x2_seed54')
MODELS = ['rmt', 'grt']
WANDB = True
TOTAL_SECONDS = 15 * 3600
MODEL_SECONDS = 7 * 3600
RUN_DIR.mkdir(parents=True, exist_ok=True)
BUDGET_FILE = RUN_DIR / 'execution.json'
budget = json.loads(BUDGET_FILE.read_text()) if BUDGET_FILE.exists() else {
    'started_at': time.time(), 'used': {}, 'executions': [], 'active': None,
}
assert (REPO_DIR / 'configs/kaggle.yaml').exists(), '수정본 checkout이 필요합니다.'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

In [ ]:
def save_budget():
    temporary = BUDGET_FILE.with_suffix('.tmp')
    temporary.write_text(json.dumps(budget, indent=2))
    temporary.replace(BUDGET_FILE)

# 커널 중단으로 남은 실행은 보수적으로 사용 시간에 포함합니다.
if budget['active']:
    active = budget['active']
    used = min(time.time() - active['started_at'], active['limit'])
    budget['used'][active['name']] = budget['used'].get(active['name'], 0) + used
    budget['executions'].append({**active, 'seconds': used, 'status': 'kernel_interrupted'})
    budget['active'] = None
save_budget()

def remaining(name):
    overall = TOTAL_SECONDS - (time.time() - budget['started_at'])
    return max(0, min(overall, MODEL_SECONDS - budget['used'].get(name, 0)))

def run_bounded(command, name, log_path, limit):
    limit = min(limit, remaining(name)) - 15  # 종료·저장을 위한 여유
    if limit <= 0:
        print(f'{name}: 실행 시간 예산 소진')
        return False
    started = time.time()
    budget['active'] = {'name': name, 'started_at': started, 'limit': limit, 'command': command}
    save_budget()
    process = None
    status = 'failed'
    def terminate():
        if process is not None and process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
    try:
        with open(log_path, 'a', buffering=1) as output, open(log_path) as reader:
            reader.seek(0, 2)
            process = subprocess.Popen(command, cwd=REPO_DIR, stdout=output,
                stderr=subprocess.STDOUT, start_new_session=True)
            while process.poll() is None:
                chunk = reader.read()
                if chunk:
                    print(chunk, end='', flush=True)
                if time.time() - started >= limit:
                    status = 'time_budget'
                    terminate()
                    break
                time.sleep(.25)
            print(reader.read(), end='', flush=True)
            if status != 'time_budget':
                status = 'completed' if process.returncode == 0 else 'failed'
    except BaseException:
        status = 'interrupted'
        raise
    finally:
        terminate()
        elapsed = time.time() - started
        budget['used'][name] = budget['used'].get(name, 0) + elapsed
        budget['executions'].append({**budget['active'], 'seconds': elapsed,
                                    'status': status, 'log': str(log_path)})
        budget['active'] = None
        save_budget()
    print(f'{name}: {status} | {log_path}')
    return status == 'completed'

In [ ]:
installation = [sys.executable, '-m', 'pip', 'install', '-e',
                str(REPO_DIR) + ('[wandb]' if WANDB else '')]
assert run_bounded(installation, 'setup', RUN_DIR / 'installation.txt', 600)
sys.path.insert(0, str(REPO_DIR / 'src'))
import torch, yaml
from grt.config import load_config, validate_config
assert torch.cuda.device_count() == 2, 'Kaggle GPU T4×2를 선택하세요.'
for index in range(2):
    print(index, torch.cuda.get_device_name(index),
          round(torch.cuda.get_device_properties(index).total_memory / 2**30, 1), 'GiB')
if WANDB:
    import wandb
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api)
SETTINGS = RUN_DIR / 'settings.yaml'
SETTINGS.write_text(yaml.safe_dump({'wandb': {'enabled': WANDB}}))
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())
print('Outputs:', RUN_DIR)

In [ ]:
# 두 모델을 먼저 측정하고, 동일한 curriculum 범위를 확정한 뒤 학습합니다.
launcher = [sys.executable, '-m', 'torch.distributed.run', '--standalone',
            '--nproc_per_node=2', 'scripts/train.py']
config_paths, measurements = {}, {}
for model in MODELS:
    paths = [str(REPO_DIR / p) for p in ['configs/base.yaml', f'configs/{model}.yaml', 'configs/kaggle.yaml']]
    if model == 'grt':
        paths.append(str(REPO_DIR / 'configs/kaggle_grt.yaml'))
    paths.append(str(SETTINGS))
    config_paths[model] = paths
    preflight = RUN_DIR / f'{model}_preflight'
    if not (preflight / 'benchmark.json').exists():
        ok = run_bounded(launcher + ['--config', *paths, '--output-dir', str(preflight),
            '--device', 'cuda', '--benchmark-steps', '3', '--memory-fraction', '0.8'],
            model, RUN_DIR / f'{model}_preflight.txt', 600)
        if not ok:
            continue
    measurements[model] = json.loads((preflight / 'benchmark.json').read_text())
assert set(measurements) == set(MODELS), '두 모델의 사전 측정이 필요합니다. preflight TXT를 확인하세요.'

PLAN_FILE = RUN_DIR / 'experiment_plan.json'
if PLAN_FILE.exists():
    plan = json.loads(PLAN_FILE.read_text())  # 재개 시 동일 설정 유지
    assert plan['models'] == MODELS
else:
    from grt.benchmark import plan_curriculum
    plan = plan_curriculum(measurements, {model: remaining(model) for model in MODELS},
                           TOTAL_SECONDS - (time.time() - budget['started_at']))
    assert plan['stage_count'] >= 2, '원본 예산으로2쌍까지 끝낼 수 없는 측정치입니다. benchmark.json을 확인하세요.'
    PLAN_FILE.write_text(json.dumps(plan, indent=2))
print('공통 실험 계획:', plan)
for model in MODELS:
    calibrated = RUN_DIR / f'{model}_calibrated.yaml'
    override = {'training': {'curriculum': measurements[model]['calibrated_override']['training']['curriculum'][:plan['stage_count']]}}
    calibrated.write_text(yaml.safe_dump(override, sort_keys=False))
    configs = config_paths[model] + [str(calibrated)]
    cfg = load_config(configs, output_dir=RUN_DIR / model)
    validate_config(cfg)
    print(model, [(s.num_pairs, s.max_steps, s.batch_size, s.grad_accum_steps) for s in cfg.training.curriculum])
    run_bounded(launcher + ['--config', *configs, '--output-dir', str(RUN_DIR / model),
        '--device', 'cuda'], model, RUN_DIR / f'{model}_console.txt', remaining(model))
print('총 경과 시간:', round((time.time() - budget['started_at']) / 3600, 2), '시간')


In [ ]:
import matplotlib.pyplot as plt
for model in MODELS:
    root = RUN_DIR / model
    if not (root / 'summary.json').exists():
        print(model, '완료된 summary 없음; execution.json / console.txt 확인')
        continue
    summary = json.loads((root / 'summary.json').read_text())
    print(model, (root / 'summary.txt').read_text())
    for name in ['evaluation', 'length_evaluation']:
        if (root / f'{name}.txt').exists():
            print((root / f'{name}.txt').read_text())
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    offset = 0
    for index, stage in enumerate(summary['stages'], 1):
        directory = root / f"stage_{index:02d}_pairs{stage['num_pairs']}_key{stage['key_size']}"
        metrics = directory / 'metrics.jsonl'
        rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()] if metrics.exists() else []
        # checkpoint 재개로 중복된 step은 마지막 기록을 사용.
        rows = sorted({row['step']: row for row in rows}.values(), key=lambda row: row['step'])
        steps = [offset + row['step'] for row in rows]
        for axis, metric in zip(axes, ['loss', 'exact_match']):
            axis.plot(steps, [row['validation']['fixed_length'][metric] for row in rows],
                      label=f"{stage['num_pairs']} pairs")
        offset += stage['step']
    axes[0].set_ylabel('fixed-length validation CE (value + EOS)')
    axes[1].set_ylabel('generated exact match (value + EOS)')
    axes[1].axhline(.99, color='gray', linestyle='--')
    axes[1].set_ylim(0, 1.03)
    for axis in axes:
        axis.set_xlabel('total optimizer updates')
        axis.grid(alpha=.2)
        axis.legend()
    fig.tight_layout()
    fig.savefig(root / 'convergence.png', dpi=150)
    plt.show()
print('공유: execution.json, experiment_plan.json, 각 모델의 preflight/benchmark.json, summary/evaluation/length_evaluation.json,')
print('convergence.png 및 각 stage의 metrics.jsonl, resolved_config.yaml, metadata.json')